# Vignette

Example workflow to convert CITE-seq data to the FCS file format and back, without losing metadata.

This example uses the 5k PBMC NextGEM dataset from 10x Genomics.

Download data from: http://cf.10xgenomics.com/samples/cell-exp/3.1.0/5k_pbmc_protein_v3_nextgem/5k_pbmc_protein_v3_nextgem_filtered_feature_bc_matrix.h5

In [1]:
import io
import os

import flowio
import numpy as np
import scanpy as sc

from anndata_fcs import anndata_to_fcs, fcs_to_anndata, read_fcs, write_fcs

DATA_DIR = os.path.join(os.getcwd(), "..", "..", "data")

### Load AnnData object from raw data

In [2]:
adata = sc.read_10x_h5(
    filename=os.path.join(DATA_DIR, "5k_pbmc_protein_v3_nextgem_filtered_feature_bc_matrix.h5"),
    gex_only=False,
)
adata.var_names_make_unique()

/tmp/ipykernel_20062/654629035.py:1: FutureWarning: Import from `scanpy.io` instead
  adata = sc.read_10x_h5(
/home/malte/Dokumente/Github/anndata-fcs/.venv/lib/python3.14/site-packages/scanpy/io/_read.py:303: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  return AnnData(matrix, obs=obs_dict, var=var_dict)


### Filter CITE-seq data and add annotations

In [3]:
adata_citeseq = adata[:, adata.var["feature_types"] == "Antibody Capture"].copy()

# Use the antibody target as marker label ($PnS)
adata_citeseq.var["pns"] = adata_citeseq.var["gene_ids"]

# Annotations stored in obs are embedded in the FCS file
adata_citeseq.obs["total_counts"] = np.asarray(adata_citeseq.X.sum(axis=1)).ravel()
adata_citeseq

AnnData object with n_obs × n_vars = 5527 × 32
    obs: 'total_counts'
    var: 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence', 'pns'
    layers: None (.X)

### Write FCS file

`write_fcs` stores `X` as event data, `var_names` as channel names (`$PnN`) and the `pns` column as marker labels
(`$PnS`). `obs`, `obs_names`, other `var` columns and `uns` are embedded as `ANNDATA_FCS_*` keywords. A synthetic
`barcode_rank` channel holds the position of each cell, so gated events can be mapped back to their barcodes.

In [4]:
fcs_path = os.path.join(DATA_DIR, "citeseq.fcs")
write_fcs(adata_citeseq, fcs_path)

### Read FCS file

In [5]:
adata_restored = read_fcs(fcs_path)
adata_restored

AnnData object with n_obs × n_vars = 5527 × 32
    obs: 'total_counts'
    var: 'pne', 'png', 'pnr', 'pns', 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'
    uns: 'fcs'
    layers: None (.X)

In [6]:
adata_restored.obs.head()

,total_counts
AAACCCACAGGCTTGC-1,4761.0
AAACCCAGTAGTTAGA-1,4010.0
AAACGAAGTAACGATA-1,1824.0
AAACGAAGTGGATCAG-1,3144.0
AAACGAATCATGAGAA-1,3118.0


In [7]:
adata_restored.var[["pns", "pnr", "feature_types"]].head()

,pns,pnr,feature_types
CD3_TotalSeqB,CD3,262144,Antibody Capture
CD4_TotalSeqB,CD4,262144,Antibody Capture
CD8a_TotalSeqB,CD8a,262144,Antibody Capture
CD11b_TotalSeqB,CD11b,262144,Antibody Capture
CD14_TotalSeqB,CD14,262144,Antibody Capture


### Recover cells from a gated FCS file

Events exported from a gate in flow cytometry software keep the `barcode_rank` channel. When reading such a file,
the observations of the gated events are restored. Here, a gate on CD3 and CD4 is simulated with NumPy.

In [8]:
fdata = anndata_to_fcs(adata_citeseq)
events = fdata.as_array(preprocess=False)

cd3 = events[:, fdata.pnn_labels.index("CD3_TotalSeqB")]
cd4 = events[:, fdata.pnn_labels.index("CD4_TotalSeqB")]
in_gate = (cd3 > 90) & (cd3 < 5000) & (cd4 > 200) & (cd4 < 5000)

# Export gated events with the metadata of the original file
structural = set(flowio.fcs_keywords.FCS_STANDARD_REQUIRED_KEYWORDS)
metadata = {key: value for key, value in fdata.text.items() if key not in structural}
handle = io.BytesIO()
flowio.create_fcs(
    handle,
    events[in_gate].ravel(),
    fdata.pnn_labels,
    opt_channel_names=fdata.pns_labels,  # flowio ignores $PnS in metadata_dict
    metadata_dict=metadata,
)
handle.seek(0)

adata_gated = fcs_to_anndata(flowio.FlowData(handle))
adata_gated

AnnData object with n_obs × n_vars = 2581 × 32
    obs: 'total_counts'
    var: 'pne', 'png', 'pnr', 'pns', 'gene_ids', 'feature_types', 'genome', 'pattern', 'read', 'sequence'
    uns: 'fcs'
    layers: None (.X)

In [9]:
# Barcodes and annotations of the gated cells are restored
assert (adata_gated.obs_names == adata_citeseq.obs_names[in_gate]).all()
adata_gated.obs.head()

,total_counts
AAACGAATCATGAGAA-1,3118.0
AAACGAATCGACACTA-1,2821.0
AAACGCTAGAGTAACT-1,1748.0
AAACGCTAGGATAATC-1,2429.0
AAACGCTGTACTGCCG-1,1091.0


### Read FCS files from cytometers

All keywords of an FCS file are kept: per-channel keywords as `var` columns, global keywords in `uns["fcs"]["text"]`.
Writing the AnnData object back to FCS keeps the event data and keywords.

In [10]:
adata_cytometer = read_fcs(os.path.join(DATA_DIR, "HTS_BD_LSR_II_Mixed_Specimen_001_D6_D06.fcs"))
adata_cytometer.var

,pnr,pne,pnv,png,pndisplay,pnbs,pnms
FSC-A,262144,"0,0",611,1.0,LOG,-1,0
FSC-H,262144,"0,0",611,1.0,LOG,-1,0
FSC-W,262144,"0,0",611,1.0,<NA>,-1,0
SSC-A,262144,"0,0",210,1.0,LOG,-1,0
SSC-H,262144,"0,0",210,1.0,LOG,-1,0
SSC-W,262144,"0,0",210,1.0,<NA>,-1,0
FITC-A,262144,"0,0",580,1.0,LOG,-1,0
PerCP-Cy5-5-A,262144,"0,0",580,1.0,LOG,-1,0
AmCyan-A,262144,"0,0",550,1.0,LOG,-1,0
PE-TxRed YG-A,262144,"0,0",500,1.0,LOG,-1,0


In [11]:
{key: adata_cytometer.uns["fcs"]["text"][key] for key in ["cyt", "date", "spill"]}

{'cyt': 'LSRII',
 'date': '26-OCT-2012',
 'spill': '4,FITC-A,PerCP-Cy5-5-A,AmCyan-A,PE-TxRed YG-A,1,0,0.021999996191960042,0,0,1,0,0,0.15999999857599995,0,1,0,0.03999999613600002,0,0.019999999516400014,1'}